# QEDark Validation

DMeRates QEDark form factors compared with QCDark rate integration using a QEDark adapter.

In [ ]:
import os
import sys
import warnings

sys.path.insert(0, '..')
sys.path.insert(0, '.')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

import numpy as np
import matplotlib.pyplot as plt
import numericalunits as nu
from scipy.integrate import simpson

nu.reset_units('SI')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torchquad')
plt.rcParams['text.usetex'] = False
plt.rcParams['figure.dpi'] = 120

BENCHMARK_MASSES_MEV = [10.0, 100.0, 1000.0]
BENCHMARK_FDM = {'heavy': 0, 'light': 2}
DEFAULT_NE_BINS = [1, 2, 3, 4, 5]

from tests.validation_helpers import QEDarkFFAdapter, load_upstream_qcdark, set_dmerates_to_qcdark_astro
from tests.conftest import hdf5_path
from DMeRates.DMeRate import DMeRate

qcdark = load_upstream_qcdark()
plt.rcParams['text.usetex'] = False

In [ ]:
def _ratio(candidate, reference):
    ratio = np.full_like(reference, np.nan, dtype=float)
    mask = np.isfinite(reference) & (reference > np.max(reference) * 1e-10)
    ratio[mask] = candidate[mask] / reference[mask]
    return ratio, mask


def plot_spectrum_overlay(energy, dmrates, reference, title, reference_label='upstream'):
    ratio, mask = _ratio(dmrates, reference)
    fig, (ax, rx) = plt.subplots(2, 1, figsize=(7, 5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    ax.plot(energy, dmrates, lw=2, label='DMeRates')
    ax.plot(energy, reference, '--', lw=2, label=reference_label)
    ax.set_yscale('log')
    ax.set_ylabel('events/kg/year/eV')
    ax.set_title(title)
    ax.legend()
    rx.plot(energy, ratio, color='black', lw=1.5)
    rx.fill_between(energy, 0.95, 1.05, color='tab:green', alpha=0.18)
    rx.axhline(1.0, color='0.4', lw=1)
    rx.set_ylim(0.5, 1.5)
    rx.set_xlabel('energy [eV]')
    rx.set_ylabel('ratio')
    plt.show()
    if np.any(mask):
        total_dm = simpson(dmrates, x=energy)
        total_ref = simpson(reference, x=energy)
        max_resid = np.nanmax(np.abs(ratio[mask] - 1.0))
        print(f'total rate (DMeRates): {total_dm:.6e} events/kg/year')
        print(f'total rate (upstream): {total_ref:.6e} events/kg/year')
        print(f'rel. difference: {(total_dm / total_ref - 1.0):+.3%}')
        print(f'max bin residual: {max_resid:.3%}')

In [ ]:
raw = np.loadtxt(hdf5_path('form_factors/QEDark/Si_f2.txt'), skiprows=1)
fcrys = np.transpose(np.resize(raw, (500, 900)))
adapter = QEDarkFFAdapter('Si')
dmr = DMeRate('Si', form_factor_type='qedark', device='cpu')
q_index, e_index = 8, 64
expected = fcrys[q_index, e_index] * (2.0 / 137.0) / 4.0
print('raw value:', fcrys[q_index, e_index])
print('expected wk/4 value:', expected)
print('adapter value:', adapter.ff[q_index, e_index])
print('DMeRates value:', dmr.form_factor.ff[q_index, e_index])
assert np.isclose(adapter.ff[q_index, e_index], expected)
assert np.isclose(dmr.form_factor.ff[q_index, e_index], expected)

In [ ]:
def compare_qedark_spectrum(material, mass_mev=100.0, fdm_n=0):
    adapter = QEDarkFFAdapter(material)
    energy_ref, spectrum_ref = qcdark.d_rate(
        mass_mev * 1e6,
        adapter,
        FDM_exp=fdm_n,
        screening=qcdark.default_no_sreen,
        astro_model=qcdark.default_astro,
    )
    obj = DMeRate(material, form_factor_type='qedark', device='cpu')
    set_dmerates_to_qcdark_astro(obj, qcdark)
    energy_dm, spectra_dm = obj.calculate_spectrum([mass_mev], 'imb', fdm_n, integrate=True, DoScreen=False)
    if not np.allclose(energy_dm, energy_ref):
        print(f'Grid offset for {material}: DMeRates starts at {energy_dm[0]:.3f} eV, adapter starts at {energy_ref[0]:.3f} eV')
        spectrum_ref = np.interp(energy_dm, energy_ref, spectrum_ref, left=0.0, right=0.0)
        energy_ref = energy_dm
    plot_spectrum_overlay(energy_dm, spectra_dm[0], spectrum_ref, f'QEDark {material}, mX={mass_mev:g} MeV, FDMn={fdm_n}')
    return obj, energy_dm, spectra_dm[0]

obj_si_qe, e_qe, s_qe = compare_qedark_spectrum('Si', 100.0, 0)
obj_ge_qe, _, _ = compare_qedark_spectrum('Ge', 100.0, 2)

In [ ]:
qcdark_obj = DMeRate('Si', form_factor_type='qcdark', device='cpu')
set_dmerates_to_qcdark_astro(qcdark_obj, qcdark)
e_qc, s_qc = qcdark_obj.calculate_spectrum([100.0], 'imb', 0, integrate=True, DoScreen=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(e_qc, s_qc[0], label='DMeRates QCDark')
ax.plot(e_qe, s_qe, label='DMeRates QEDark')
ax.set_yscale('log')
ax.set_xlabel('energy [eV]')
ax.set_ylabel('events/kg/year/eV')
ax.set_title('QCDark vs QEDark form-factor families')
ax.legend()
plt.show()
print('QCDark total:', simpson(s_qc[0], x=e_qc))
print('QEDark total:', simpson(s_qe, x=e_qe))

In [ ]:
def compare_qedark_ne(material='Si', fdm_n=0):
    masses = np.array(BENCHMARK_MASSES_MEV)
    obj = DMeRate(material, form_factor_type='qedark', device='cpu')
    set_dmerates_to_qcdark_astro(obj, qcdark)
    obj.change_to_step()
    rates_dm = obj.calculate_rates(masses, 'imb', fdm_n, DEFAULT_NE_BINS, integrate=True, DoScreen=False)
    rates_dm = (rates_dm * nu.kg * nu.year).detach().cpu().numpy()

    adapter = QEDarkFFAdapter(material)
    e2q = {'Si': 3.8, 'Ge': 3.0}[material]
    rates_ref = []
    for mass in masses:
        _, bins = qcdark.d_rate_FanoQ(
            mass * 1e6,
            adapter,
            e2q,
            FDM_exp=fdm_n,
            screening=qcdark.default_no_sreen,
            astro_model=qcdark.default_astro,
        )
        rates_ref.append(bins[1:6])
    rates_ref = np.array(rates_ref).T

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, ne in enumerate(DEFAULT_NE_BINS):
        ax.plot(masses, rates_dm[i], marker='o', label=f'DMeRates ne={ne}')
        ax.plot(masses, rates_ref[i], '--', color=ax.lines[-1].get_color(), label=f'Adapter ne={ne}')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('DM mass [MeV]')
    ax.set_ylabel('events/kg/year')
    ax.set_title(f'QEDark {material} dR/dne sweep, FDMn={fdm_n}')
    ax.legend(ncol=2, fontsize=8)
    plt.show()
    print('Residuals at 100 MeV:', rates_dm[:, 1] / rates_ref[:, 1] - 1.0)

compare_qedark_ne('Si', 0)
compare_qedark_ne('Ge', 2)